In [2]:
import pandas as pd
import duckdb
data = [
    # Device A
    ["A", "2026-07-01 08:00:00", "NORMAL"],
    ["A", "2026-07-01 08:05:00", "NORMAL"],
    ["A", "2026-07-01 08:10:00", "ERROR"],
    ["A", "2026-07-01 08:15:00", "ERROR"],
    ["A", "2026-07-01 08:20:00", "NORMAL"],

    # Device B
    ["B", "2026-07-01 08:00:00", "NORMAL"],
    ["B", "2026-07-01 08:05:00", "ERROR"],
    ["B", "2026-07-01 08:10:00", "NORMAL"],
    ["B", "2026-07-01 08:15:00", "NORMAL"],

    # Device C
    ["C", "2026-07-01 08:00:00", "ERROR"],
    ["C", "2026-07-01 08:05:00", "ERROR"],
    ["C", "2026-07-01 08:10:00", "ERROR"],
    ["C", "2026-07-01 08:15:00", "NORMAL"],
]

df = pd.DataFrame(
    data,
    columns=["device_id", "collect_time", "status"]
)

df["collect_time"] = pd.to_datetime(df["collect_time"])

print(df)



   device_id        collect_time  status
0          A 2026-07-01 08:00:00  NORMAL
1          A 2026-07-01 08:05:00  NORMAL
2          A 2026-07-01 08:10:00   ERROR
3          A 2026-07-01 08:15:00   ERROR
4          A 2026-07-01 08:20:00  NORMAL
5          B 2026-07-01 08:00:00  NORMAL
6          B 2026-07-01 08:05:00   ERROR
7          B 2026-07-01 08:10:00  NORMAL
8          B 2026-07-01 08:15:00  NORMAL
9          C 2026-07-01 08:00:00   ERROR
10         C 2026-07-01 08:05:00   ERROR
11         C 2026-07-01 08:10:00   ERROR
12         C 2026-07-01 08:15:00  NORMAL


## 题目要求

### 分别使用 SQL 和 Pandas 完成：

- 找出每个设备中，状态发生变化的记录。

### 最终输出字段：

- `device_id`
- `collect_time`
- `status`
- `previous_status`
- `is_status_changed`


In [7]:
# SQL轨道

query = """

WITH previous_status_table AS (
    SELECT
        device_id,
        collect_time,
        status,
        LAG(status) OVER(
            PARTITION BY device_id
            ORDER BY collect_time
        ) AS previous_status
    FROM df
),

status_compare AS (
    SELECT
        device_id,
        collect_time,
        status,
        previous_status,
        CASE
            WHEN previous_status IS NOT NULL
             AND status <> previous_status
            THEN True
            ELSE False
        END AS is_status_changed
    FROM previous_status_table
)

SELECT
    *
FROM status_compare

ORDER BY device_id, collect_time;
"""
df_sql = duckdb.execute(query).fetchdf()
df_sql

,device_id,collect_time,status,previous_status,is_status_changed
0,A,2026-07-01 08:00:00,NORMAL,NaN,False
1,A,2026-07-01 08:05:00,NORMAL,NORMAL,False
2,A,2026-07-01 08:10:00,ERROR,NORMAL,True
3,A,2026-07-01 08:15:00,ERROR,ERROR,False
4,A,2026-07-01 08:20:00,NORMAL,ERROR,True
5,B,2026-07-01 08:00:00,NORMAL,NaN,False
6,B,2026-07-01 08:05:00,ERROR,NORMAL,True
7,B,2026-07-01 08:10:00,NORMAL,ERROR,True
8,B,2026-07-01 08:15:00,NORMAL,NORMAL,False
9,C,2026-07-01 08:00:00,ERROR,NaN,False


In [16]:
df_pd = (
    df
    .sort_values(by=['device_id', 'collect_time'])
    .assign(
        previous_status=lambda x: (
            x.groupby('device_id')['status'].shift(1)
        ),
        is_status_changed=lambda x: (
            x['previous_status'].notna()
            & (x['status'] != x['previous_status'])
        )
    )
    .loc[lambda x: x['is_status_changed']]
    .reset_index(drop=True)
)
df_pd

,device_id,collect_time,status,previous_status,is_status_changed
0,A,2026-07-01 08:10:00,ERROR,NORMAL,True
1,A,2026-07-01 08:20:00,NORMAL,ERROR,True
2,B,2026-07-01 08:05:00,ERROR,NORMAL,True
3,B,2026-07-01 08:10:00,NORMAL,ERROR,True
4,C,2026-07-01 08:15:00,NORMAL,ERROR,True
